In [1]:
import findspark
findspark.init()

In [2]:
from pyspark.sql import SparkSession

In [9]:
spark = SparkSession.builder.appName("customer code").master('local[3]').config('spark.driver.host','127.0.0.1')\
.config('spark.ui.port',0).config('spark.shuffle.partitions',10).getOrCreate()

In [10]:
spark

In [11]:
PM = spark.read.format('csv')\
.option('header',True).option('inferSchema',True)\
.load(r'/home/AzureAdmin/Dataset_01/Baseband_02/Dataset_01_LTE_2100.csv')

In [12]:
PM.rdd.getNumPartitions()

3

In [25]:
pm=PM.repartition(5,'Sector')

In [20]:
pm.rdd.getNumPartitions()

5

In [16]:
pm1= pm.coalesce(4)

In [17]:
pm1.rdd.getNumPartitions()

4

In [18]:
PM.printSchema()

root
 |-- Base station: string (nullable = true)
 |-- Sector: integer (nullable = true)
 |-- Timestamp: timestamp (nullable = true)
 |-- Radio unit energy consumption: double (nullable = true)
 |-- Baseband energy consumption: double (nullable = true)
 |-- 4G max active users DL: double (nullable = true)
 |-- 4G max active users UL: double (nullable = true)
 |-- 4G data volume DL: double (nullable = true)
 |-- 4G data volume UL: double (nullable = true)
 |-- 4G max RRC users: double (nullable = true)
 |-- 4G RB utilization: double (nullable = true)
 |-- 4G CQI rank 1: double (nullable = true)
 |-- 4G CQI rank 2: double (nullable = true)
 |-- 4G CQI rank 3: double (nullable = true)
 |-- 4G CQI rank 4: double (nullable = true)
 |-- 4G RRC users: double (nullable = true)
 |-- 4G active users UL: double (nullable = true)
 |-- 4G active users DL: double (nullable = true)
 |-- 4G MIMO rank DL: double (nullable = true)



In [21]:
ppl= spark.read.format('json').load(r'/home/AzureAdmin/data/people.json')

In [22]:
acc_par = spark.read.format('parquet').load(r'/home/AzureAdmin/data/accounts_parquet/accounts_parquet')

In [24]:
ppl.join(acc_par,ppl.pcode==acc_par.zipcode).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [pcode#216], [zipcode#228], Inner, BuildLeft, false
   :- BroadcastExchange HashedRelationBroadcastMode(List(input[2, string, false]),false), [plan_id=194]
   :  +- Filter isnotnull(pcode#216)
   :     +- FileScan json [age#214L,name#215,pcode#216] Batched: false, DataFilters: [isnotnull(pcode#216)], Format: JSON, Location: InMemoryFileIndex(1 paths)[file:/home/AzureAdmin/data/people.json], PartitionFilters: [], PushedFilters: [IsNotNull(pcode)], ReadSchema: struct<age:bigint,name:string,pcode:string>
   +- Filter isnotnull(zipcode#228)
      +- FileScan parquet [acct_num#220,acct_create_dt#221L,acct_close_dt#222L,first_name#223,last_name#224,address#225,city#226,state#227,zipcode#228,phone_number#229,created#230L,modified#231L] Batched: true, DataFilters: [isnotnull(zipcode#228)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/AzureAdmin/data/accounts_parquet/accounts_parquet], PartitionFilt

In [27]:
pm.select('Sector').distinct().show()

+------+
|Sector|
+------+
|     3|
|     4|
|     1|
|     2|
+------+



In [32]:
PM.where(pm.Sector >2 ).write.format('noop').mode('append').save(r'/home/AzureAdmin/data/nov2')

In [33]:
PM.where(pm.Sector >2 ).explain()

== Physical Plan ==
*(1) Filter (isnotnull(Sector#92) AND (Sector#92 > 2))
+- FileScan csv [Base station#91,Sector#92,Timestamp#93,Radio unit energy consumption#94,Baseband energy consumption#95,4G max active users DL#96,4G max active users UL#97,4G data volume DL#98,4G data volume UL#99,4G max RRC users#100,4G RB utilization#101,4G CQI rank 1#102,4G CQI rank 2#103,4G CQI rank 3#104,4G CQI rank 4#105,4G RRC users#106,4G active users UL#107,4G active users DL#108,4G MIMO rank DL#109] Batched: false, DataFilters: [isnotnull(Sector#92), (Sector#92 > 2)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/AzureAdmin/Dataset_01/Baseband_02/Dataset_01_LTE_2100.csv], PartitionFilters: [], PushedFilters: [IsNotNull(Sector), GreaterThan(Sector,2)], ReadSchema: struct<Base station:string,Sector:int,Timestamp:timestamp,Radio unit energy consumption:double,Ba...




In [36]:
PM.select('Sector','Base station','Timestamp').write.format('noop').mode('append').save(r'/home/AzureAdmin/data/nov2')

In [37]:
PM.select('Sector','Base station','Timestamp').show()

+------+------------+-------------------+
|Sector|Base station|          Timestamp|
+------+------------+-------------------+
|     1|     Site 36|2023-10-08 06:00:00|
|     2|     Site 36|2023-10-08 06:00:00|
|     3|     Site 36|2023-10-08 06:00:00|
|     1|     Site 36|2023-10-23 14:30:00|
|     2|     Site 36|2023-10-23 14:30:00|
|     3|     Site 36|2023-10-23 14:30:00|
|     1|     Site 36|2023-10-20 09:30:00|
|     2|     Site 36|2023-10-20 09:30:00|
|     3|     Site 36|2023-10-20 09:30:00|
|     1|     Site 36|2023-10-20 12:15:00|
|     2|     Site 36|2023-10-20 12:15:00|
|     3|     Site 36|2023-10-20 12:15:00|
|     1|     Site 36|2023-10-11 23:45:00|
|     2|     Site 36|2023-10-11 23:45:00|
|     3|     Site 36|2023-10-11 23:45:00|
|     1|     Site 36|2023-10-15 09:45:00|
|     2|     Site 36|2023-10-15 09:45:00|
|     3|     Site 36|2023-10-15 09:45:00|
|     1|     Site 36|2023-10-13 22:30:00|
|     2|     Site 36|2023-10-13 22:30:00|
+------+------------+-------------

In [44]:
from pyspark.storagelevel import StorageLevel

In [45]:
PM.persist(StorageLevel.DISK_ONLY)
grp=PM.groupBy('Sector')

res = grp.count()
res.show()


#PM->grp->res

+------+-----+
|Sector|count|
+------+-----+
|     1|50343|
|     3|48565|
|     2|50343|
|     4| 1775|
+------+-----+



In [42]:
mxdata = grp.max()
mxdata.show()

#grp->mxdata

+------+-----------+----------------------------------+--------------------------------+---------------------------+---------------------------+----------------------+----------------------+---------------------+----------------------+------------------+------------------+------------------+------------------+-----------------+-----------------------+-----------------------+--------------------+
|Sector|max(Sector)|max(Radio unit energy consumption)|max(Baseband energy consumption)|max(4G max active users DL)|max(4G max active users UL)|max(4G data volume DL)|max(4G data volume UL)|max(4G max RRC users)|max(4G RB utilization)|max(4G CQI rank 1)|max(4G CQI rank 2)|max(4G CQI rank 3)|max(4G CQI rank 4)|max(4G RRC users)|max(4G active users UL)|max(4G active users DL)|max(4G MIMO rank DL)|
+------+-----------+----------------------------------+--------------------------------+---------------------------+---------------------------+----------------------+----------------------+------------

In [43]:
PM.unpersist()

DataFrame[Base station: string, Sector: int, Timestamp: timestamp, Radio unit energy consumption: double, Baseband energy consumption: double, 4G max active users DL: double, 4G max active users UL: double, 4G data volume DL: double, 4G data volume UL: double, 4G max RRC users: double, 4G RB utilization: double, 4G CQI rank 1: double, 4G CQI rank 2: double, 4G CQI rank 3: double, 4G CQI rank 4: double, 4G RRC users: double, 4G active users UL: double, 4G active users DL: double, 4G MIMO rank DL: double]